# Geração de curso a partir de demanda bruta — demonstraçãoPega texto como o cliente manda — e-mail encaminhado, conversa de WhatsApp, transcrição dereunião — e mostra o núcleo da plataforma transformando isso em curso estruturado.Importa o código real do `ai-structuring-service`. Nada aqui é reimplementado para ademonstração: é o mesmo caso de uso que a rota `POST /api/v1/structuring` expõe, com o mesmoprompt versionado, a mesma validação de schema e o mesmo tratamento de falha.## O que é demonstrado, e o que depende de chave de APIQuatro das cinco provas abaixo são **determinísticas**: rodam em qualquer máquina, sem chave esem custo. Não são simulação — são propriedades do núcleo sendo exercitadas de verdade, com aresposta do modelo fixada para que o caminho exercitado seja sempre o mesmo.| Prova | Precisa de chave? | O que mostra ||---|---|---|| 1. Geração completa | **sim**, para gerar de verdade | o curso montado a partir do texto || 2. A IA não inventa | não | campo sem base no texto vira lacuna declarada || 3. Valor sem base é descartado | não | o núcleo não aproveita palpite, nem se o modelo insistir || 4. Saída fora do schema é recusada | não | a barreira que impede dado malformado de entrar || 5. Falha não perde a demanda | não | timeout e indisponibilidade não custam o texto |Sem `OPENROUTER_API_KEY`, a prova 1 roda com o provedor mock e diz isso explicitamente: você vêo encadeamento funcionando, mas o curso exibido é a resposta fixa do mock, não geração real.> **Sobre medição de qualidade:** este notebook demonstra **comportamento**, não mede acurácia.> A medição contra gabarito anotado é outro notebook, `avaliacao-de-modelos.ipynb`, e depende de> um conjunto de casos reais anotados por humano que o projeto ainda não tem.

## PreparoImporta o pacote do serviço direto da árvore do repositório. Para rodar:```bashcd services/ai-structuring-service && pip install -e ".[dev]"pip install jupyterlab ipywidgetscd ../../notebooks && jupyter lab demonstracao-geracao-de-curso.ipynb```Para a prova 1 gerar de verdade, exporte a chave **antes** de abrir o Jupyter:`export OPENROUTER_API_KEY=...` (ou `$env:OPENROUTER_API_KEY=...` no PowerShell). A chave nuncaé escrita no notebook.

In [1]:
import json
import os
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "services" / "ai-structuring-service"))

os.environ.setdefault("ENVIRONMENT", "development")
os.environ.setdefault("LOG_LEVEL", "ERROR")
os.environ.setdefault("AI_STRUCTURING_PORT", "8003")
os.environ.setdefault("PIPELINE_SERVICE_URL", "http://localhost:8001")
os.environ.setdefault("LLM_PROVIDER", "mock")
os.environ.setdefault("LLM_MODEL", "mock-demo")
os.environ.setdefault("LLM_API_KEY", "")
os.environ.setdefault("LLM_TIMEOUT_SECONDS", "60")
os.environ.setdefault("LLM_MAX_RETRIES", "2")
os.environ.setdefault("LLM_TEMPERATURE", "0")

from pydantic import SecretStr

from app.core.config import get_settings
from app.core.exceptions import LLMTimeoutError
from app.domain.course import CAMPOS_EXTRAIDOS, CAMPOS_GERADOS
from app.providers.http_llm_provider import HttpLLMProvider
from app.providers.mock_provider import MockLLMProvider
from app.services.structuring_service import RawDemand, StructuringService

CONFIG = get_settings()
CHAVE = os.environ.get("OPENROUTER_API_KEY", "").strip()

# Modelo usado na prova 1 quando ha chave. Troque pelo slug que quiser testar.
MODELO = "openai/gpt-4o-mini"

print("nucleo importado de services/ai-structuring-service")
print("campos extraidos do texto :", ", ".join(CAMPOS_EXTRAIDOS))
print("campos gerados            :", ", ".join(CAMPOS_GERADOS))
print("provedor da prova 1       :", f"OpenRouter ({MODELO})" if CHAVE else "mock (sem OPENROUTER_API_KEY)")

nucleo importado de services/ai-structuring-service
campos extraidos do texto : tema, nicho, publico_alvo, numero_participantes, carga_horaria, formato
campos gerados            : objetivos_aprendizagem, ementa
provedor da prova 1       : mock (sem OPENROUTER_API_KEY)


## A função que você vai usar`estruturar(texto)` faz o caminho inteiro: monta o prompt versionado, chama o modelo, valida asaída contra o schema canônico e imprime o resultado campo a campo, com as lacunas destacadas.É a única função que você precisa para testar qualquer texto seu, mais abaixo.

In [2]:
LARGURA = 80
MARCA_DE_DESCARTE = "valor descartado na validacao"


def _config_openrouter(slug):
    """Reaproveita o provedor HTTP existente apontado para o OpenRouter.

    O OpenRouter fala o dialeto /chat/completions da OpenAI, entao nao ha
    provedor novo: muda a base, o modelo e a credencial.
    """
    return CONFIG.model_copy(
        update={
            "llm_provider": "http",
            "llm_base_url": "https://openrouter.ai/api/v1",
            "llm_completions_path": "/chat/completions",
            "llm_model": slug,
            "llm_api_key": SecretStr(CHAVE),
        }
    )


def _provedor(resposta=None, falha=None):
    """Monta o provedor da chamada.

    Resposta ou falha informadas forcam o mock: e assim que as provas
    deterministicas exercitam um caminho especifico do nucleo.
    """
    if resposta is not None or falha is not None:
        return MockLLMProvider(model="mock-demo", response_text=resposta, fail_with=falha)
    if not CHAVE:
        return MockLLMProvider(model="mock-demo")
    return HttpLLMProvider(_config_openrouter(MODELO))


def _valor_simples(valor):
    """Desembrulha enum para o texto que o contrato publica."""
    return getattr(valor, "value", valor)


def _linha(rotulo, valor):
    print(f"  {rotulo:<26} {_valor_simples(valor)}")


def mostrar(curso):
    """Imprime o curso estruturado campo a campo, com as lacunas a vista."""
    lacunas = {g.campo: g for g in curso.campos_ausentes}

    print("CURSO ESTRUTURADO")
    print("-" * LARGURA)
    for campo in CAMPOS_EXTRAIDOS:
        if campo in lacunas:
            _linha(campo, f"-- LACUNA ({lacunas[campo].tipo.value}) --")
        else:
            valor = getattr(curso, campo)
            _linha(campo, "(vazio)" if valor in (None, "") else _valor_simples(valor))

    print()
    print("  objetivos de aprendizagem")
    for i, objetivo in enumerate(curso.objetivos_aprendizagem, 1):
        print(f"    {i}. {objetivo}")
    if not curso.objetivos_aprendizagem:
        print("    (nenhum)")

    print()
    print("  ementa")
    for modulo in curso.ementa:
        carga = f"{modulo.carga_horaria}h" if modulo.carga_horaria else "carga nao definida"
        print(f"    {modulo.titulo}  ({carga})")
        for topico in modulo.topicos:
            print(f"      - {topico}")
    if not curso.ementa:
        print("    (nenhum modulo)")

    if curso.campos_ausentes:
        print()
        print(f"  A CONFIRMAR COM O CLIENTE ({len(curso.campos_ausentes)})")
        for g in curso.campos_ausentes:
            print(f"    {g.campo} [{g.tipo.value}]")
            print(f"      {g.motivo or 'sem motivo registrado'}")

    descartes = [o for o in curso.observacoes if MARCA_DE_DESCARTE in o]
    outras = [o for o in curso.observacoes if MARCA_DE_DESCARTE not in o]
    if outras:
        print()
        print("  observacoes do modelo")
        for obs in outras:
            print(f"    - {obs}")
    if descartes:
        print()
        print(f"  DESCARTES FEITOS PELA VALIDACAO ({len(descartes)})")
        for obs in descartes:
            print(f"    - {obs}")


async def estruturar(texto, *, resposta=None, falha=None, silencioso=False):
    """Estrutura um texto de demanda e devolve o desfecho.

    Args:
        texto: a demanda bruta, como o cliente mandou.
        resposta: forca o modelo a devolver este texto (provas deterministicas).
        falha: forca o modelo a falhar com este erro (prova 5).
        silencioso: nao imprime; util quando a celula quer formatar por conta.
    """
    servico = StructuringService(_provedor(resposta, falha), CONFIG)
    desfecho = await servico.structure_course(RawDemand(demand_id="demo", text=texto))

    if silencioso:
        return desfecho

    print(f"prompt: {desfecho.prompt_id}   tentativas: {desfecho.attempts}", end="")
    if desfecho.completion is not None:
        uso = desfecho.completion.usage
        print(
            f"   modelo: {desfecho.completion.model}"
            f"   tokens: {uso.prompt_tokens} entrada / {uso.completion_tokens} saida"
            f"   {desfecho.elapsed_ms:.0f} ms"
        )
    else:
        print(f"   {desfecho.elapsed_ms:.0f} ms")
    print("=" * LARGURA)

    if desfecho.course is not None:
        mostrar(desfecho.course)
    else:
        print("SEM CURSO VALIDADO")
        print("-" * LARGURA)
        _linha("erro", type(desfecho.error).__name__ if desfecho.error else "desconhecido")
        _linha("mensagem", str(desfecho.error) if desfecho.error else "-")
        _linha("vale repetir?", desfecho.retryable)
        print()
        print("  A DEMANDA BRUTA NAO FOI PERDIDA:")
        _linha("demand_id", desfecho.demand.demand_id)
        _linha("texto preservado", f"{len(desfecho.demand.text)} caracteres, intactos")
    return desfecho


print("pronto: use `await estruturar(seu_texto)`")

pronto: use `await estruturar(seu_texto)`


---## Prova 1 — geração completa a partir de um e-mailTexto como o cliente manda, com cabeçalho de encaminhamento, assinatura e informação espalhadaem meio a conversa.

In [3]:
EMAIL = """
---------- Mensagem encaminhada ---------
De: [contato do cliente]
Data: 14 de outubro de 2026
Assunto: ENC: treinamento pra equipe de manutencao

Boa tarde, tudo bem?

Conforme conversamos na reuniao de ontem, precisamos de um treinamento de NR-12 para o pessoal
da manutencao da planta 2. Sao uns 25 tecnicos. A ideia e fazer em dois dias de 8 horas, no
proprio site, porque tirar o time da planta por mais tempo complica a producao.

O foco tem que ser pratico: bloqueio e etiquetagem, protecao de maquina e o que fazer quando
encontram um equipamento sem protecao.

Fico no aguardo da proposta.

Atenciosamente,
[nome do contato]
"""

_ = await estruturar(EMAIL)

prompt: extract-requirements.v3   tentativas: 1   modelo: mock-demo   tokens: 3644 entrada / 374 saida   0 ms
CURSO ESTRUTURADO
--------------------------------------------------------------------------------
  tema                       Seguranca do trabalho em ambiente industrial
  nicho                      Industria metalurgica
  publico_alvo               Operadores de chao de fabrica e lideres de turno
  numero_participantes       -- LACUNA (ambigua) --
  carga_horaria              16
  formato                    presencial

  objetivos de aprendizagem
    1. Identificar riscos ocupacionais tipicos da operacao metalurgica
    2. Aplicar o procedimento de bloqueio e etiquetagem antes da manutencao
    3. Reagir a um incidente seguindo o plano de emergencia da planta

  ementa
    Fundamentos de seguranca ocupacional  (4h)
      - Normas aplicaveis
      - Mapa de riscos
      - Equipamentos de protecao
    Riscos da operacao metalurgica  (6h)
      - Calor e ruido
      - Moviment

---## Prova 2 — a IA não inventa o que não está no textoEsta é a afirmação central do projeto, e a que mais precisa de prova.A conversa abaixo **nega explicitamente** a carga horária — *"a gente ainda nao fechou quantashoras"* — e não diz nada sobre formato nem sobre o setor da empresa.Um modelo que preenchesse esses três campos estaria alucinando: entregando suposição com cara derequisito confirmado, que depois vira proposta comercial errada. O comportamento correto édeclará-los como lacuna, com o motivo que o operador consegue levar de volta ao cliente.

In [4]:
WHATSAPP = """
[10:02] oi! seguinte, a diretoria pediu um treinamento de lideranca pros coordenadores novos
[10:02] sao 12 pessoas que foram promovidas esse ano
[10:03] eles querem falar de feedback, gestao de conflito e como conduzir reuniao de equipe
[10:05] a gente ainda nao fechou quantas horas, depende do orcamento
"""

HONESTA = json.dumps(
    {
        "tema": "Lideranca para coordenadores recem-promovidos",
        "nicho": None,
        "publico_alvo": "Coordenadores promovidos no ano corrente",
        "numero_participantes": 12,
        "carga_horaria": None,
        "formato": None,
        "objetivos_aprendizagem": [
            "Dar e receber feedback de forma estruturada",
            "Mediar conflito dentro da equipe",
            "Conduzir reuniao de equipe com objetivo e encaminhamentos",
        ],
        "ementa": [
            {"titulo": "Feedback", "topicos": ["Feedback descritivo", "Conversa dificil"]},
            {"titulo": "Gestao de conflito", "topicos": ["Origem do conflito", "Mediacao"]},
            {"titulo": "Conducao de reuniao", "topicos": ["Pauta", "Encaminhamentos"]},
        ],
        "campos_ausentes": [
            {
                "campo": "carga_horaria",
                "tipo": "ausente",
                "motivo": "o cliente diz que ainda nao fechou a carga horaria, que depende do orcamento",
            },
            {
                "campo": "formato",
                "tipo": "ausente",
                "motivo": "o texto nao diz se o treinamento e presencial, online ou hibrido",
            },
            {
                "campo": "nicho",
                "tipo": "ausente",
                "motivo": "o setor de atuacao da empresa nao aparece na conversa",
            },
        ],
        "observacoes": ["Carga horaria e formato precisam de confirmacao antes da proposta."],
    },
    ensure_ascii=False,
)

_ = await estruturar(WHATSAPP, resposta=HONESTA)

prompt: extract-requirements.v3   tentativas: 1   modelo: mock-demo   tokens: 3563 entrada / 272 saida   0 ms
CURSO ESTRUTURADO
--------------------------------------------------------------------------------
  tema                       Lideranca para coordenadores recem-promovidos
  nicho                      -- LACUNA (ausente) --
  publico_alvo               Coordenadores promovidos no ano corrente
  numero_participantes       12
  carga_horaria              -- LACUNA (ausente) --
  formato                    -- LACUNA (ausente) --

  objetivos de aprendizagem
    1. Dar e receber feedback de forma estruturada
    2. Mediar conflito dentro da equipe
    3. Conduzir reuniao de equipe com objetivo e encaminhamentos

  ementa
    Feedback  (carga nao definida)
      - Feedback descritivo
      - Conversa dificil
    Gestao de conflito  (carga nao definida)
      - Origem do conflito
      - Mediacao
    Conducao de reuniao  (carga nao definida)
      - Pauta
      - Encaminhamentos

 

### O contraste: o que aconteceria se o modelo chutasseMesma entrada, resposta diferente. Agora o modelo preenche 16 horas, "presencial" e "Industria",que **não estão no texto**, e não declara lacuna nenhuma.Repare no desfecho: o núcleo **aceita** esta resposta. Ela é bem-formada, passa no schema, e semlacuna declarada não há nada para reconciliar.É exatamente por isso que validação de schema não basta sozinha. O schema pega dado**malformado**; dado **inventado e bem-formado** só a comparação com a fonte pega — e é esse otrabalho das métricas de alucinação e da revisão humana na tela.

In [5]:
CHUTADA = json.loads(HONESTA)
CHUTADA["carga_horaria"] = 16
CHUTADA["formato"] = "presencial"
CHUTADA["nicho"] = "Industria"
CHUTADA["campos_ausentes"] = []
CHUTADA["observacoes"] = []

desfecho = await estruturar(WHATSAPP, resposta=json.dumps(CHUTADA, ensure_ascii=False))

print()
print("=" * LARGURA)
print("POR QUE ISTO E UM PROBLEMA")
print("-" * LARGURA)
print("  A resposta passou no schema e nao declarou lacuna nenhuma.")
print("  Estes tres campos foram entregues como se fossem requisito do cliente:")
for campo in ("carga_horaria", "formato", "nicho"):
    valor = _valor_simples(getattr(desfecho.course, campo))
    print(f"    {campo:<22} {valor!r:<16} <- nao esta no texto")
print()
print("  O schema nao tem como pegar isto: a resposta e bem-formada.")
print("  Quem pega sao as metricas de alucinacao, comparando com a fonte,")
print("  e o revisor humano na tela de revisao (RF14).")

prompt: extract-requirements.v3   tentativas: 1   modelo: mock-demo   tokens: 3563 entrada / 169 saida   0 ms
CURSO ESTRUTURADO
--------------------------------------------------------------------------------
  tema                       Lideranca para coordenadores recem-promovidos
  nicho                      Industria
  publico_alvo               Coordenadores promovidos no ano corrente
  numero_participantes       12
  carga_horaria              16
  formato                    presencial

  objetivos de aprendizagem
    1. Dar e receber feedback de forma estruturada
    2. Mediar conflito dentro da equipe
    3. Conduzir reuniao de equipe com objetivo e encaminhamentos

  ementa
    Feedback  (carga nao definida)
      - Feedback descritivo
      - Conversa dificil
    Gestao de conflito  (carga nao definida)
      - Origem do conflito
      - Mediacao
    Conducao de reuniao  (carga nao definida)
      - Pauta
      - Encaminhamentos

POR QUE ISTO E UM PROBLEMA
-------------------

---## Prova 3 — valor sem base é descartado, não só sinalizadoAqui está a regra mais forte do núcleo, e ela é fácil de passar batido.A transcrição abaixo diz **doze** pessoas em um momento e corrige para **oito** no fim. O modelofaz o que parece razoável: declara a contradição *e* escolhe oito, a correção mais recente.O núcleo **joga a escolha fora**. Campo com lacuna declarada não fica com valor nenhum, mesmoque o modelo tenha preenchido, e a validação registra por quê. Escolher entre trechos emconflito é decisão do revisor, não do modelo — e a plataforma não deixa o palpite passar porrequisito confirmado.Veja, na saída, `numero_participantes` virando lacuna e o descarte aparecendo registrado.

In [6]:
TRANSCRICAO = """
[00:03] Entao, a gente ta montando a area de dados aqui e os analistas vieram tudo de outras areas.
[00:21] Sao oito pessoas, nenhuma com base de programacao.
[00:40] O que a gente precisa e Python voltado pra analise, bem pratico, mexendo em planilha de verdade.
[01:05] Falei com a diretoria e eles toparam 40 horas, podendo quebrar em modulos.
[01:30] Ah, e tem que ser remoto, porque metade do time ta em outra cidade.
[01:52] Uma coisa: em outro momento voces falaram que o time tinha doze pessoas. Hoje sao oito.
"""

CONTRADITORIA = json.dumps(
    {
        "tema": "Python para analise de dados",
        "nicho": None,
        "publico_alvo": "Analistas vindos de outras areas, sem base de programacao",
        "numero_participantes": 8,
        "carga_horaria": 40,
        "formato": "online",
        "objetivos_aprendizagem": [
            "Manipular dados tabulares com Python",
            "Automatizar o tratamento de planilha",
            "Produzir analise descritiva a partir de dados reais",
        ],
        "ementa": [
            {
                "titulo": "Python essencial para analise",
                "topicos": ["Tipos e estruturas", "Fluxo de controle"],
                "carga_horaria": 12,
            },
            {
                "titulo": "Dados tabulares",
                "topicos": ["Leitura de planilha", "Limpeza"],
                "carga_horaria": 16,
            },
            {
                "titulo": "Analise descritiva",
                "topicos": ["Agregacao", "Visualizacao basica"],
                "carga_horaria": 12,
            },
        ],
        "campos_ausentes": [
            {
                "campo": "numero_participantes",
                "tipo": "contraditoria",
                "motivo": "a fonte cita doze pessoas em um trecho e oito em outro",
            },
            {
                "campo": "nicho",
                "tipo": "ausente",
                "motivo": "o setor da empresa nao aparece; 'area de dados' e o time interno, nao o nicho de mercado",
            },
        ],
        "observacoes": ["Confirmar o numero de participantes antes de dimensionar turma e custo."],
    },
    ensure_ascii=False,
)

desfecho = await estruturar(TRANSCRICAO, resposta=CONTRADITORIA)

print()
print("=" * LARGURA)
print("O QUE O NUCLEO FEZ COM A ESCOLHA DO MODELO")
print("-" * LARGURA)
proposto = json.loads(CONTRADITORIA)["numero_participantes"]
print(f"  o modelo devolveu       numero_participantes = {proposto}")
print(f"  o curso validado tem    numero_participantes = {desfecho.course.numero_participantes!r}")
print()
print("  O valor nao sobreviveu a validacao, e isso e proposital: o proprio")
print("  modelo declarou a informacao contraditoria na fonte. Carga horaria e")
print("  formato, que a fonte afirma sem conflito, passaram intactos.")

prompt: extract-requirements.v3   tentativas: 1   modelo: mock-demo   tokens: 3616 entrada / 270 saida   0 ms
CURSO ESTRUTURADO
--------------------------------------------------------------------------------
  tema                       Python para analise de dados
  nicho                      -- LACUNA (ausente) --
  publico_alvo               Analistas vindos de outras areas, sem base de programacao
  numero_participantes       -- LACUNA (contraditoria) --
  carga_horaria              40
  formato                    online

  objetivos de aprendizagem
    1. Manipular dados tabulares com Python
    2. Automatizar o tratamento de planilha
    3. Produzir analise descritiva a partir de dados reais

  ementa
    Python essencial para analise  (12h)
      - Tipos e estruturas
      - Fluxo de controle
    Dados tabulares  (16h)
      - Leitura de planilha
      - Limpeza
    Analise descritiva  (12h)
      - Agregacao
      - Visualizacao basica

  A CONFIRMAR COM O CLIENTE (2)
    nich

---## Prova 4 — saída fora do schema é recusada, não aceita "quase certa"A barreira que a ADR-0006 declara: a resposta do modelo é validada contra a forma canônica**antes** de virar resultado, e resposta fora do contrato é falha — não resultado parcial, não"dá pra aproveitar".Abaixo, seis respostas defeituosas, uma por tipo de defeito. Cada uma é montada sobre a resposta**completa e sem lacuna declarada** da célula anterior, de propósito: se partíssemos da respostacom lacunas, a regra da prova 3 descartaria o campo defeituoso antes de a validação de tipochegar nele, e o teste mediria a regra errada.A verificação não é por inspeção: o `assert` no fim da célula quebra a execução do notebook sequalquer defeito passar. Ou seja, esta página não consegue ser publicada com a garantiaquebrada.Repare também nas linhas de log acima da tabela — é o registro de diagnóstico do RNF05, quenomeia o **campo** que violou o contrato (`campos=preco_sugerido`) e o tamanho da resposta, semdevolver o texto recusado.

In [7]:
# Base completa, com todos os campos preenchidos e nenhuma lacuna declarada:
# e sobre ela que cada defeito e introduzido, um por vez.
COMPLETA = json.loads(json.dumps(CHUTADA))

SEM_TEMA = {k: v for k, v in COMPLETA.items() if k != "tema"}

DEFEITOS = [
    ("JSON malformado", "{isto nao e json valido"),
    ("conversa em volta do objeto", 'Claro! Aqui esta:\n\n{"tema": "Lideranca"}'),
    ("chave fora do contrato", {**COMPLETA, "preco_sugerido": 4800}),
    ("campo obrigatorio ausente", SEM_TEMA),
    ("tipo trocado (carga como texto)", {**COMPLETA, "carga_horaria": "dezesseis"}),
    ("valor fora do dominio (formato)", {**COMPLETA, "formato": "metaverso"}),
]

resultados = []
for rotulo, defeito in DEFEITOS:
    resposta = defeito if isinstance(defeito, str) else json.dumps(defeito, ensure_ascii=False)
    d = await estruturar(WHATSAPP, resposta=resposta, silencioso=True)
    resultados.append((rotulo, d.course is None, type(d.error).__name__ if d.error else "-"))

print()
print(f"{'defeito':<36} {'recusado?':<12} erro")
print("-" * LARGURA)
for rotulo, recusado, erro in resultados:
    print(f"{rotulo:<36} {('SIM' if recusado else 'NAO (!!)'):<12} {erro}")

passaram = [rotulo for rotulo, recusado, _ in resultados if not recusado]
print()
print(f"{len(resultados) - len(passaram)} de {len(resultados)} recusadas.")
if passaram:
    print("DEFEITO QUE PASSOU: " + ", ".join(passaram))
else:
    print("Nenhuma resposta defeituosa virou curso.")

assert not passaram, f"garantia de schema quebrada para: {passaram}"

Resposta do provedor recusada pelo schema do curso estruturado (demand_id=demo, prompt=extract-requirements.v3, tentativas=1, campos=(raiz), caracteres=23).


Resposta do provedor recusada pelo schema do curso estruturado (demand_id=demo, prompt=extract-requirements.v3, tentativas=1, campos=(raiz), caracteres=40).


Resposta do provedor recusada pelo schema do curso estruturado (demand_id=demo, prompt=extract-requirements.v3, tentativas=1, campos=preco_sugerido, caracteres=700).


Resposta do provedor recusada pelo schema do curso estruturado (demand_id=demo, prompt=extract-requirements.v3, tentativas=1, campos=tema, caracteres=619).


Resposta do provedor recusada pelo schema do curso estruturado (demand_id=demo, prompt=extract-requirements.v3, tentativas=1, campos=carga_horaria, caracteres=685).


Resposta do provedor recusada pelo schema do curso estruturado (demand_id=demo, prompt=extract-requirements.v3, tentativas=1, campos=formato, caracteres=675).



defeito                              recusado?    erro
--------------------------------------------------------------------------------
JSON malformado                      SIM          LLMInvalidResponseError
conversa em volta do objeto          SIM          LLMInvalidResponseError
chave fora do contrato               SIM          LLMInvalidResponseError
campo obrigatorio ausente            SIM          LLMInvalidResponseError
tipo trocado (carga como texto)      SIM          LLMInvalidResponseError
valor fora do dominio (formato)      SIM          LLMInvalidResponseError

6 de 6 recusadas.
Nenhuma resposta defeituosa virou curso.


---## Prova 5 — falha do modelo não custa a demanda do clienteRegra dura do projeto: a demanda bruta é persistida pelo `ingestion-service` **antes** dequalquer chamada ao modelo. Timeout, indisponibilidade ou estouro de cota não podem fazer ooperador perder o texto que acabou de colar.Note as tentativas gastas: timeout é marcado como retentável, então a política de retentativainsiste antes de desistir.

In [8]:
_ = await estruturar(EMAIL, falha=LLMTimeoutError("o provedor nao respondeu em 60 s"))

prompt: extract-requirements.v3   tentativas: 3   1520 ms
SEM CURSO VALIDADO
--------------------------------------------------------------------------------
  erro                       LLMTimeoutError
  mensagem                   o provedor nao respondeu em 60 s
  vale repetir?              True

  A DEMANDA BRUTA NAO FOI PERDIDA:
  demand_id                  demo
  texto preservado           635 caracteres, intactos


---# Teste você mesmoTroque o texto abaixo pelo que quiser — e-mail de cliente, conversa, transcrição, qualquer coisa— e rode a célula. Com `OPENROUTER_API_KEY` no ambiente, a chamada vai ao modelo de verdade; semela, você vê o encadeamento com a resposta fixa do mock.

In [9]:
MEU_TEXTO = """
Oi, tudo bem? A gente precisa de um treinamento de Excel avancado pro time comercial.
Sao 18 pessoas, a maioria usa planilha todo dia mas ninguem sabe tabela dinamica nem
procv direito. Queria algo de umas 20 horas, online, porque o time ta espalhado em tres
estados. O que precisa sair de la: a pessoa consegue montar um relatorio de vendas sozinha.
"""

_ = await estruturar(MEU_TEXTO)

prompt: extract-requirements.v3   tentativas: 1   modelo: mock-demo   tokens: 3574 entrada / 374 saida   0 ms
CURSO ESTRUTURADO
--------------------------------------------------------------------------------
  tema                       Seguranca do trabalho em ambiente industrial
  nicho                      Industria metalurgica
  publico_alvo               Operadores de chao de fabrica e lideres de turno
  numero_participantes       -- LACUNA (ambigua) --
  carga_horaria              16
  formato                    presencial

  objetivos de aprendizagem
    1. Identificar riscos ocupacionais tipicos da operacao metalurgica
    2. Aplicar o procedimento de bloqueio e etiquetagem antes da manutencao
    3. Reagir a um incidente seguindo o plano de emergencia da planta

  ementa
    Fundamentos de seguranca ocupacional  (4h)
      - Normas aplicaveis
      - Mapa de riscos
      - Equipamentos de protecao
    Riscos da operacao metalurgica  (6h)
      - Calor e ruido
      - Moviment

### Comparar modelos no mesmo textoCom chave, roda o mesmo texto em vários modelos e põe o resultado lado a lado. É a versãointerativa da comparação que o `avaliacao-de-modelos.ipynb` faz de forma sistemática, comgabarito e métrica.

In [10]:
MODELOS_PARA_COMPARAR = [
    "openai/gpt-4o-mini",
    "anthropic/claude-sonnet-4.5",
]

if not CHAVE:
    print("Sem OPENROUTER_API_KEY: comparar modelos exige chamada real.")
    print("Com a chave no ambiente, esta celula roda MEU_TEXTO em cada modelo da lista.")
else:
    for slug in MODELOS_PARA_COMPARAR:
        config = _config_openrouter(slug)
        servico = StructuringService(HttpLLMProvider(config), config)
        d = await servico.structure_course(RawDemand(demand_id="comparacao", text=MEU_TEXTO))

        print("=" * LARGURA)
        print(f"MODELO: {slug}")
        print("-" * LARGURA)
        if d.course is None:
            print(f"  falhou: {type(d.error).__name__}: {d.error}")
        else:
            c = d.course
            _linha("tema", c.tema)
            _linha("carga horaria", c.carga_horaria)
            _linha("formato", c.formato)
            _linha("participantes", c.numero_participantes)
            _linha("modulos na ementa", len(c.ementa))
            _linha("objetivos", len(c.objetivos_aprendizagem))
            _linha("lacunas declaradas", len(c.campos_ausentes))
            for g in c.campos_ausentes:
                print(f"      {g.campo} [{g.tipo.value}]")
            _linha("latencia", f"{d.elapsed_ms:.0f} ms")
        print()

Sem OPENROUTER_API_KEY: comparar modelos exige chamada real.
Com a chave no ambiente, esta celula roda MEU_TEXTO em cada modelo da lista.


### Interface com campo de texto (opcional)Se `ipywidgets` estiver instalado, a célula abaixo dá uma caixa de texto e um botão — útil parademonstrar ao vivo, sem editar célula. Sem a biblioteca, ela diz como instalar e não falha.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
except ImportError:
    print("ipywidgets nao instalado. Para a interface interativa:")
    print("  pip install ipywidgets")
    print("Sem ela, use a celula 'Teste voce mesmo' acima, editando MEU_TEXTO.")
else:
    import asyncio

    entrada = widgets.Textarea(
        value=MEU_TEXTO.strip(),
        placeholder="Cole aqui o e-mail, a conversa ou a transcricao do cliente",
        layout=widgets.Layout(width="100%", height="180px"),
    )
    botao = widgets.Button(description="Estruturar curso", button_style="primary")
    saida = widgets.Output()

    async def _estruturar_da_caixa():
        with saida:
            clear_output()
            texto = entrada.value.strip()
            if not texto:
                print("Cole um texto de demanda antes de estruturar.")
                return
            print("estruturando...")
            clear_output(wait=True)
            await estruturar(texto)

    botao.on_click(lambda _: asyncio.ensure_future(_estruturar_da_caixa()))
    display(widgets.VBox([entrada, botao, saida]))

---## O que esta demonstração sustenta**Sustenta, sem depender de chave nem de dado de cliente:**- o encadeamento completo do núcleo — prompt versionado, chamada ao modelo, validação de schema,  política de retentativa, tratamento de falha — rodando sobre texto heterogêneo em formato  (e-mail encaminhado, conversa, transcrição);- que resposta fora do contrato é **recusada** nos seis tipos de defeito, e não aceita como  resultado parcial — verificado por `assert`, não por inspeção;- que lacuna na fonte é **declarada e classificada** em ausente, ambígua ou contraditória, com  motivo legível para o operador;- que valor preenchido em campo com lacuna declarada é **descartado** pela validação, com o  descarte registrado — o palpite do modelo não chega ao revisor com cara de requisito;- que falha do modelo **não custa a demanda bruta** do cliente.**Não sustenta:**- **qualidade da geração**, que depende de chamada real a modelo e de julgamento humano sobre os  cinco campos de texto livre (`tema`, `nicho`, `publico_alvo`, `objetivos_aprendizagem`,  `ementa`);- **acurácia medida**, que exige o conjunto de casos reais anonimizados com gabarito anotado — o  projeto ainda não tem esses casos, e nenhum código resolve isso;- **comparação entre modelos com número**, que é o `avaliacao-de-modelos.ipynb` e depende do  mesmo conjunto.O par de células da prova 2 é o limite mais importante de entender: **validação de schema pegadado malformado, não dado inventado.** Uma resposta que chuta carga horária e formato passa noschema sem reclamar. O que pega isso é a métrica de alucinação comparada à fonte e a revisãohumana na tela — e é por isso que as duas existem no projeto, em vez de confiar no schema.